# Glacial Lake Mapping in Southeast Iceland using Sentinel-1 and Sentinel-2 via OpenEO

Based on OBIA workflow by [Dabiri et al. 2021](https://austriaca.at/0xc1aa5576%200x003c9b50.pdf).


**Workflow per year (2016–2025):**
1. Load Sentinel-2 summer composite via OpenEO
2. Load Sentinel-1 summer composite via OpenEO 
3. Compute S2 indices (NDVI, NDWI, NDSI, brightness)
4. Compute S1 features (VV_dB, VH_dB, mean_dB, VV/VH ratio)
5. Segment image using SLIC on combined S1+S2 feature stack
6. Classify water segments using S2 NDWI OR S1 backscatter threshold + refinement
7. Export polygon outlines per year as GeoPackage

**Data used:**
- 2016-2025: Sentinel-2 (`SENTINEL2_L2A`)
- 2016-2025: Sentinel-1 (`SENTINEL1_GRD`)
- Copernicus Global 30 meter DEM (`COPERNICUS_30`) (terrain correction for S1)


Imports

In [71]:
import os
import json
import numpy as np
import rasterio
import shutil
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from rasterio.features import shapes as rio_shapes
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform
import geopandas as gpd
from shapely.geometry import shape, mapping
from skimage.segmentation import slic
from skimage.measure import regionprops
from scipy.ndimage import uniform_filter  # for basic speckle smoothing
import openeo
from tqdm import tqdm
import warnings
from config import OUT_DIR, DATA_DIR, OBIA_DIR
warnings.filterwarnings("ignore")

### Configuration

In [72]:
# Bounding box: Southeast Iceland (Vatnajökull glacier lake area)
BBOX = {
    "west":  -16.5,
    "south":  63.97,
    "east":  -16.1,
    "north":  64.2,
}

BBOX_EPSG = 4326

# Reproject to UTM zone 27N (EPSG:32627) 
TARGET_CRS = "EPSG:32627"

# Temporal range: summer months per year
YEARS = list(range(2026, 2027))  # 2016–2026 inclusive
SUMMER_START_MM_DD = "06-20"
SUMMER_END_MM_DD   = "06-22"

# Cloud masking
MAX_CLOUD_COVER = 50  # percent

Segmentation and classification parameters

In [73]:
SEG_PARAMS = {             # parameters for SLIC segmentation
    "n_segments":  800,   
    "compactness": 0.4,   
    "sigma":       0.8,
}

# S2 classification thresholds
NDWI_WATER_THRESHOLD = 0.2   # NDWI >= threshold → water 
NDVI_VEGE_THRESHOLD  = 0.1   # NDVI < threshold to exclude vegetation patches

# S1 classification threshold
S1_MEAN_WATER_THRESHOLD_DB = -18.0  # Mean backscatter (VH+VV)/2 < -threshold dB → water

# Area filters
MIN_LAKE_AREA_M2 = 10_000    
MAX_LAKE_AREA_M2 = 35_000_000  # upper bound to exclude sea/ocean

# S1 feature weight in SLIC stack relative to S2 features
# Values 0.5–1.5: lower = S2 dominates segmentation, higher = S1 drives boundaries
S1_FEATURE_WEIGHT = 1.5

# Fusion mode:
# "OR"  – classify as water if NDWI≥threshold OR S1_mean<threshold
# "AND" – both must agree (more conservative, fewer false positives)
# "S2"  – S2 only 
# "S1"  – S1 only 
FUSION_MODE = "AND"

### OpenEO connection

In [74]:
def connect_openeo(backend_url: str = "https://openeo.dataspace.copernicus.eu") -> openeo.Connection:
    """
    Connect and authenticate to the Copernicus Dataspace OpenEO backend.
    """
    print(f"Connecting to OpenEO backend: {backend_url}")
    conn = openeo.connect(backend_url)
    conn.authenticate_oidc()  # Opens browser for OIDC login on first run
    print("✓ Connected and authenticated.")
    return conn

In [75]:
# Check connection and list collections
conn = connect_openeo()
conn.list_collections()

Connecting to OpenEO backend: https://openeo.dataspace.copernicus.eu
Authenticated using refresh token.
✓ Connected and authenticated.


[{'description': 'Sentinel 3 imagery captured by OLCI sensor',
  'extent': {'spatial': {'bbox': [[-180.0, -85.0, 180.0, 85.0]]},
   'temporal': {'interval': [['2016-04-17T11:33:13Z', None]]}},
  'id': 'SENTINEL3_OLCI_L1B',
  'license': 'proprietary',
  'links': [{'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/',
    'rel': 'root',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-3-olci',
    'rel': 'self',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections',
    'rel': 'parent',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-3-olci/queryables',
    'rel': 'http://www.opengis.net/def/rel/ogc/1.0/queryables',
    'type': 'application/schema+json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-3-olci/items',
    'rel': 'items',
    'type': 'application/geo+json'}],
  'providers': [],
  'stac_extensions': ['https://stac-extensions.github.io/scientific/v1.0.0/schema.json',
   'https://stac-extensions.github.io/sat/v1.0.0/schema.json',
   'https://stac-extensions.github.io/eo/v1.0.0/schema.json'],
  'stac_version': '1.0.0',
  'title': 'Sentinel 3 OLCI'},
 {'description': 'Sentinel 3 imagery captured by SLSTR sensor',
  'extent': {'spatial': {'bbox': [[-180.0, -85.0, 180.0, 85.0]]},
   'temporal': {'interval': [['2016-04-17T11:33:13Z', None]]}},
  'id': 'SENTINEL3_SLSTR',
  'license': 'proprietary',
  'links': [{'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/',
    'rel': 'root',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-3-slstr',
    'rel': 'self',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections',
    'rel': 'parent',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-3-slstr/queryables',
    'rel': 'http://www.opengis.net/def/rel/ogc/1.0/queryables',
    'type': 'application/schema+json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-3-slstr/items',
    'rel': 'items',
    'type': 'application/geo+json'}],
  'providers': [],
  'stac_extensions': ['https://stac-extensions.github.io/scientific/v1.0.0/schema.json',
   'https://stac-extensions.github.io/sat/v1.0.0/schema.json',
   'https://stac-extensions.github.io/eo/v1.0.0/schema.json'],
  'stac_version': '1.0.0',
  'title': 'Sentinel 3 SLSTR'},
 {'description': 'Sentinel 5 Precursor imagery captured by TROPOMI sensor.\n\nThis dataset only supports loading one band at a time.',
  'extent': {'spatial': {'bbox': [[-180.0, -85.0, 180.0, 85.0]]},
   'temporal': {'interval': [['2018-04-30T00:18:50Z', None]]}},
  'id': 'SENTINEL_5P_L2',
  'license': 'proprietary',
  'links': [{'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/',
    'rel': 'root',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-5p-l2',
    'rel': 'self',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections',
    'rel': 'parent',
    'type': 'application/json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-5p-l2/queryables',
    'rel': 'http://www.opengis.net/def/rel/ogc/1.0/queryables',
    'type': 'application/schema+json'},
   {'href': 'https://sh.dataspace.copernicus.eu/catalog/v1/collections/sentinel-5p-l2/items',
    'rel': 'items',
    'type': 'application/geo+json'}],
  'providers': [],
  'stac_extensions': ['https://stac-extensions.github.io/scientific/v1.0.0/schema.json',
   'https://stac-extensions.github.io/sat/v1.0.0/schema.json',
   'https://docs.sentinel-hub.com/api/latest/stac/s5p/v1.0.0/schema.json'],
  'stac_version': '1.0.0',
  'title': 'Sentinel 5 Precursor'},
 {'description': 'The Sentinel-1 mission p

### Sentinel-2 data loading

In [76]:
def build_s2_job(conn: openeo.Connection, year: int) -> openeo.rest.datacube.DataCube:
    """
    Build OpenEO process graph for Sentinel-2 L2A summer median composite.
    Applies SCL-based cloud masking. Returns bands: B02, B03, B04, B08, B11, B12.
    """
    temporal_extent = [
        f"{year}-{SUMMER_START_MM_DD}",
        f"{year}-{SUMMER_END_MM_DD}",
    ]

    bands = ["B02", "B03", "B04", "B08", "B11", "B12", "SCL"]

    dc = conn.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=BBOX,
        temporal_extent=temporal_extent,
        bands=bands,
        max_cloud_cover=MAX_CLOUD_COVER,
    )

    # Cloud masking: exclude cloud shadow(3), cloud medium(8), cloud high(9), cirrus(10)
    scl = dc.band("SCL")
    cloud_mask = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10)
    dc = dc.mask(cloud_mask)

    dc = dc.filter_bands(["B02", "B03", "B04", "B08", "B11", "B12"])
    dc = dc.reduce_dimension(dimension="t", reducer="median")

    return dc


def download_s2_composite(conn: openeo.Connection, year: int, out_path: str) -> str:
    """Download S2 median summer composite for a given year. Caches to disk."""
    if os.path.exists(out_path):
        print(f"  ↳ [S2] Using cached composite")
        return out_path

    print(f"  ↳ [S2] Building OpenEO job for {year}...")
    dc = build_s2_job(conn, year)

    tmp_dir = os.path.join(os.path.dirname(out_path), f"_tmp_s2_{year}")
    os.makedirs(tmp_dir, exist_ok=True)

    result = dc.save_result(format="GTiff")
    job = result.create_job(title=f"glacial_lakes_S2_{year}")
    print(f"  ↳ [S2] Submitting batch job (ID: {job.job_id})...")
    job.start_and_wait()
    job.get_results().download_files(target=tmp_dir)

    downloaded_tifs = sorted([f for f in os.listdir(tmp_dir) if f.lower().endswith(".tif")])
    if not downloaded_tifs:
        raise FileNotFoundError(f"No .tif files found in {tmp_dir} for S2 year {year}.")
    if len(downloaded_tifs) > 1:
        print(f"  ⚠ [S2] Multiple TIFs found: {downloaded_tifs} — using first.")

    os.rename(os.path.join(tmp_dir, downloaded_tifs[0]), out_path)
    try:
        os.rmdir(tmp_dir)
    except OSError:
        pass

    print(f"  ✓ [S2] Downloaded")
    return out_path

### Sentinel-1 data loading

In [77]:
def build_s1_job(conn: openeo.Connection, year: int) -> openeo.rest.datacube.DataCube:
    """
    Build OpenEO process graph for Sentinel-1 GRD summer median composite.
    Returns VV and VH backscatter (sigma0 linear power units).

    The Copernicus Dataspace backend applies terrain correction server-side.
    We use a median temporal composite over the same summer window as S2.
    """
    temporal_extent = [
        f"{year}-{SUMMER_START_MM_DD}",
        f"{year}-{SUMMER_END_MM_DD}",
    ]

    dc = conn.load_collection(
        "SENTINEL1_GRD",
        spatial_extent=BBOX,
        temporal_extent=temporal_extent,
        bands=["VV", "VH"],
    )

    # Apply terrain correction using Copernicus GLO-30 DEM
    dc = dc.sar_backscatter(
        coefficient="sigma0-ellipsoid",
        elevation_model="COPERNICUS_30",
        local_incidence_angle=False,
    )

    dc = dc.reduce_dimension(dimension="t", reducer="median")
    return dc


def download_s1_composite(conn: openeo.Connection, year: int, out_path: str) -> str:
    """Download S1 median summer composite for a given year. Caches to disk."""
    if os.path.exists(out_path):
        print(f"  ↳ [S1] Using cached composite")
        return out_path

    print(f"  ↳ [S1] Building OpenEO job for {year}...")
    dc = build_s1_job(conn, year)

    tmp_dir = os.path.join(os.path.dirname(out_path), f"_tmp_s1_{year}")
    os.makedirs(tmp_dir, exist_ok=True)

    result = dc.save_result(format="GTiff")
    job = result.create_job(title=f"glacial_lakes_S1_{year}")
    print(f"  ↳ [S1] Submitting batch job (ID: {job.job_id})...")
    job.start_and_wait()
    job.get_results().download_files(target=tmp_dir)

    downloaded_tifs = sorted([f for f in os.listdir(tmp_dir) if f.lower().endswith(".tif")])
    if not downloaded_tifs:
        raise FileNotFoundError(f"No .tif files found in {tmp_dir} for S1 year {year}.")
    if len(downloaded_tifs) > 1:
        print(f"  ⚠ [S1] Multiple TIFs found: {downloaded_tifs} — using first.")

    os.rename(os.path.join(tmp_dir, downloaded_tifs[0]), out_path)
    try:
        os.rmdir(tmp_dir)
    except OSError:
        pass

    print(f"  ✓ [S1] Downloaded")
    return out_path

### S1 alignment helper

S1 and S2 are both 10 m but may have slightly different pixel grids depending on the backend's projection. This function reprojects/resamples S1 to exactly match the S2 raster grid.

In [78]:
def align_s1_to_s2(s1_path: str, s2_path: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Reproject and resample S1 to match the pixel grid of the S2.
    Returns (vv_linear, vh_linear) as float32 arrays on the S2 grid.

    Both datasets should be 10 m resolution; this handles any minor CRS or
    pixel-offset differences introduced by the OpenEO backend.
    """
    with rasterio.open(s2_path) as s2_src:
        dst_crs       = s2_src.crs
        dst_transform = s2_src.transform
        dst_width     = s2_src.width
        dst_height    = s2_src.height

    with rasterio.open(s1_path) as s1_src:
        vv_dst = np.zeros((dst_height, dst_width), dtype=np.float32)
        vh_dst = np.zeros((dst_height, dst_width), dtype=np.float32)

        reproject(
            source=rasterio.band(s1_src, 1),
            destination=vv_dst,
            src_transform=s1_src.transform,
            src_crs=s1_src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )
        reproject(
            source=rasterio.band(s1_src, 2),
            destination=vh_dst,
            src_transform=s1_src.transform,
            src_crs=s1_src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )

    # Mask fill values (0 in linear SAR data = no-data)
    vv_dst = np.where(vv_dst <= 0, np.nan, vv_dst)
    vh_dst = np.where(vh_dst <= 0, np.nan, vh_dst)

    return vv_dst, vh_dst

### Compute indices

In [79]:
def compute_s2_indices(blue, green, red, nir, swir1):
    """
    Compute Sentinel-2 spectral indices from surface reflectance (0–1 scale).
    Returns: ndvi, ndwi, ndsi, brightness (float32 arrays)
    """
    eps = 1e-10

    # NDVI = (NIR - Red) / (NIR + Red)
    ndvi = (nir - red) / (nir + red + eps)

    # NDWI = (Green - NIR) / (Green + NIR) 
    ndwi = (green - nir) / (green + nir + eps)

    # NDSI = (Green - SWIR1) / (Green + SWIR1) 
    ndsi = (green - swir1) / (green + swir1 + eps)

    # Brightness = mean of visible bands
    brightness = (blue + green + red) / 3.0

    return (
        np.clip(ndvi, -1, 1).astype(np.float32),
        np.clip(ndwi, -1, 1).astype(np.float32),
        np.clip(ndsi, -1, 1).astype(np.float32),
        brightness.astype(np.float32),
    )


def compute_s1_features(vv_linear: np.ndarray, vh_linear: np.ndarray,
                         speckle_filter_size: int = 5):
    """
    Compute Sentinel-1 derived layers from sigma0 linear power values.

    Processing steps:
      1. Apply a lightweight spatial mean filter (5*5) to further
         reduce residual speckle after temporal median compositing.
      2. Convert sigma0 from linear power to dB: 10 * log10(sigma0)
      3. Compute derived layers from the paper:
         - mean_dB  = (VV_dB + VH_dB) / 2  → primary water threshold layer
         - ratio_vv_vh = VV_dB - VH_dB      → surface roughness indicator
         - ratio_vh_vv = VH_dB - VV_dB      → double-bounce / volume scatter

    Returns: vv_db, vh_db, mean_db, ratio_vv_vh, ratio_vh_vv (float32 arrays)
    """
    eps = 1e-10

    # Spatial mean filter for speckle reduction
    # NaN-safe: replace NaN with local mean before filtering
    def nan_mean_filter(arr, size):
        mask = np.isnan(arr)
        filled = np.where(mask, 0.0, arr)
        smoothed = uniform_filter(filled, size=size)
        count = uniform_filter((~mask).astype(float), size=size)
        smoothed = np.where(count > 0, smoothed / np.maximum(count, eps), np.nan)
        return smoothed.astype(np.float32)

    vv_smooth = nan_mean_filter(vv_linear, speckle_filter_size)
    vh_smooth = nan_mean_filter(vh_linear, speckle_filter_size)

    # Convert to dB
    vv_db = (10.0 * np.log10(np.maximum(vv_smooth, eps))).astype(np.float32)
    vh_db = (10.0 * np.log10(np.maximum(vh_smooth, eps))).astype(np.float32)

    # Mask no-data pixels
    nodata = np.isnan(vv_linear) | np.isnan(vh_linear)
    vv_db[nodata] = np.nan
    vh_db[nodata] = np.nan

    # Computed layers
    mean_db      = ((vv_db + vh_db) / 2.0).astype(np.float32)
    ratio_vv_vh  = (vv_db - vh_db).astype(np.float32)  # in dB: log subtraction = linear division
    ratio_vh_vv  = (vh_db - vv_db).astype(np.float32)

    return vv_db, vh_db, mean_db, ratio_vv_vh, ratio_vh_vv

### Segmentation

In [80]:
def segment_image(blue, green, red, nir,
                   ndvi, ndwi, ndsi, swir1, brightness,
                   vv_db, vh_db, mean_db,
                   seg_params: dict,
                   s1_weight: float = 1.0) -> np.ndarray:
    """
    Segment the image using SLIC superpixels on a combined S1 + S2 feature stack.

    Feature stack:
      S2: Blue, Green, Red, SWIR1, NIR, NDVI, NDWI, NDSI, Brightness  
      S1: VV_dB, VH_dB, mean_dB                                

    All features are normalised to [0, 1] before stacking.
    S1 features are multiplied by s1_weight to control their influence on
    segment boundaries relative to S2 spectral features.

    If S1 data is unavailable (all NaN), falls back to S2-only segmentation.
    """
    def norm(arr):
        mn = np.nanpercentile(arr, 2)
        mx = np.nanpercentile(arr, 98)
        if mx == mn:
            return np.zeros_like(arr, dtype=np.float32)
        return np.clip((arr - mn) / (mx - mn), 0, 1).astype(np.float32)

    s2_layers = [
        norm(blue), norm(green), norm(red), norm(nir),
        norm(ndvi), norm(ndwi),  norm(ndsi), norm(swir1), norm(brightness),
    ]

    # Only include S1 layers where data is available
    s1_available = not np.all(np.isnan(vv_db))
    if s1_available:
        s1_layers = [
            norm(vv_db)   * s1_weight,
            norm(vh_db)   * s1_weight,
            norm(mean_db) * s1_weight,
        ]
        print("  ↳ Using S1 + S2 feature stack for segmentation.")
    else:
        s1_layers = []
        print("  ⚠ S1 data unavailable — falling back to S2-only segmentation.")

    feature_stack = np.dstack(s2_layers + s1_layers)
    feature_stack = np.nan_to_num(feature_stack, nan=0.0)

    segments = slic(
        feature_stack,
        n_segments=seg_params["n_segments"],
        compactness=seg_params["compactness"],
        sigma=seg_params["sigma"],
        start_label=1,
        channel_axis=-1,
    )

    return segments

### Classification 

In [81]:
def classify_water_segments(segments: np.ndarray,
                              ndwi: np.ndarray,
                              ndvi: np.ndarray,
                              s1_mean_db: np.ndarray,
                              seg_params: dict,
                              fusion_mode: str = "AND") -> tuple[np.ndarray, dict]:
    """
    Classify segments as water using object-based rules with S1 and S2.

    Primary water criteria:
      "OR"  : NDWI >= threshold OR  S1_mean < threshold dB  
      "AND" : NDWI >= threshold AND S1_mean < -threshold dB  
      "S2"  : NDWI >= threshold only                  
      "S1"  : S1_mean < threshold dB only             

    Refinement:
      - NDVI < threshold  (exclude vegetation)

    Returns:
      water_mask : binary uint8 array (1=water)
      stats      : dict with detection source counts for diagnostics
    """
    water_mask = np.zeros_like(segments, dtype=np.uint8)
    stats = {"s2_only": 0, "s1_only": 0, "both": 0, "rejected_vege": 0, "rejected_size": 0}

    s1_available = not np.all(np.isnan(s1_mean_db))

    props = regionprops(segments)
    for prop in props:
        seg_id = prop.label
        px_mask = segments == seg_id

        mean_ndwi = float(np.nanmean(ndwi[px_mask]))
        mean_ndvi = float(np.nanmean(ndvi[px_mask]))
        size_px   = int(np.sum(px_mask))

        is_water_s2 = mean_ndwi >= NDWI_WATER_THRESHOLD

        if s1_available:
            mean_s1 = float(np.nanmean(s1_mean_db[px_mask]))
            is_water_s1 = mean_s1 < S1_MEAN_WATER_THRESHOLD_DB
        else:
            is_water_s1 = False

        # Apply fusion mode
        if fusion_mode == "OR":
            is_water = is_water_s2 or is_water_s1
        elif fusion_mode == "AND":
            is_water = is_water_s2 and is_water_s1
        elif fusion_mode == "S2":
            is_water = is_water_s2
        elif fusion_mode == "S1":
            is_water = is_water_s1
        else:
            raise ValueError(f"Unknown fusion_mode: {fusion_mode!r}. Use 'OR', 'AND', 'S2', or 'S1'.")

        if not is_water:
            continue

        # Refinement
        if mean_ndvi >= NDVI_VEGE_THRESHOLD:
            stats["rejected_vege"] += 1
            continue

        # Track detection source for diagnostics
        if is_water_s2 and is_water_s1:
            stats["both"] += 1
        elif is_water_s2:
            stats["s2_only"] += 1
        else:
            stats["s1_only"] += 1

        water_mask[px_mask] = 1

    return water_mask, stats

### Vectorisation

In [82]:
def vectorise_water_mask(water_mask: np.ndarray,
                          transform: rasterio.transform.Affine,
                          src_crs: CRS,
                          year: int,
                          detection_stats: dict) -> gpd.GeoDataFrame:
    """
    Convert binary water mask to polygon GeoDataFrame.
    Filters by area thresholds. Reprojects to TARGET_CRS.
    Adds a 'detection_source' column for traceability.
    """
    polys = []
    for geom, val in rio_shapes(water_mask,
                                 mask=water_mask.astype(np.uint8),
                                 transform=transform):
        if val == 1:
            polys.append(shape(geom))

    if not polys:
        print(f"  ⚠ No water polygons found for {year}")
        return gpd.GeoDataFrame(columns=["geometry", "year", "area_m2"],
                                crs=src_crs)

    gdf = gpd.GeoDataFrame({"geometry": polys}, crs=src_crs)
    gdf = gdf.to_crs(TARGET_CRS)
    gdf["area_m2"] = gdf.geometry.area
    gdf["year"] = year

    # Area filter
    gdf = gdf[
        (gdf["area_m2"] >= MIN_LAKE_AREA_M2) &
        (gdf["area_m2"] <= MAX_LAKE_AREA_M2)
    ].copy().reset_index(drop=True)

    print(f"  ✓ {len(gdf)} lake polygons for {year} "
          f"(total area: {gdf['area_m2'].sum()/1e6:.2f} km²)")
    print(f"    Detection sources → S2 only: {detection_stats['s2_only']}, "
          f"S1 only: {detection_stats['s1_only']}, "
          f"Both: {detection_stats['both']}")
    print(f"    Rejected → vegetation: {detection_stats['rejected_vege']}, "
          f"too small: {detection_stats['rejected_size']}")

    return gdf

### Main pipeline

In [83]:
def process_year(conn: openeo.Connection, year: int) -> gpd.GeoDataFrame:
    """
    Workflow single year:
      1. Download S1 and S2 composites
      2. Load S2 bands
      3. Compute S2 indices
      4. Align S1 to S2 grid, compute SAR features
      5. Segment on combined feature stack
      6. Classify 
      7. Vectorise
    """
    print(f"\n{'='*60}")
    print(f"  Processing year: {year}  |  Fusion mode: {FUSION_MODE}")
    print(f"{'='*60}")

    # 1. Download composites 
    s2_path = os.path.join(DATA_DIR, f"composite_s2_{year}.tif")
    s1_path = os.path.join(DATA_DIR, f"composite_s1_{year}.tif")

    download_s2_composite(conn, year, s2_path)
    download_s1_composite(conn, year, s1_path)

    # 2. Load S2 bands
    with rasterio.open(s2_path) as src:
        # Band order from OpenEO: B02(Blue), B03(Green), B04(Red), B08(NIR), B11(SWIR1), B12(SWIR2)
        blue  = src.read(1).astype(np.float32) / 10000.0
        green = src.read(2).astype(np.float32) / 10000.0
        red   = src.read(3).astype(np.float32) / 10000.0
        nir   = src.read(4).astype(np.float32) / 10000.0
        swir1 = src.read(5).astype(np.float32) / 10000.0

        transform = src.transform
        src_crs   = src.crs

        # Mask no-data (0 in L2A)
        nodata = (blue == 0) | (nir == 0)
        for arr in [blue, green, red, nir, swir1]:
            arr[nodata] = np.nan

    print(f"  ↳ S2 image shape: {blue.shape}")

    # --- 3. Compute S2 indices ---
    print("  ↳ Computing S2 spectral indices...")
    ndvi, ndwi, ndsi, brightness = compute_s2_indices(blue, green, red, nir, swir1)

    # 4. Align S1 and compute SAR features
    print("  ↳ Aligning S1 to S2 grid...")
    try:
        vv_linear, vh_linear = align_s1_to_s2(s1_path, s2_path)
        print("  ↳ Computing S1 SAR features (dB conversion + speckle filter)...")
        vv_db, vh_db, mean_db, ratio_vv_vh, ratio_vh_vv = compute_s1_features(
            vv_linear, vh_linear, speckle_filter_size=5
        )
    except Exception as e:
        print(f"  ⚠ S1 processing failed ({e}). Falling back to S2-only.")
        vv_db = vh_db = mean_db = np.full_like(blue, np.nan)

    # 5. Segment
    print(f"  ↳ Segmenting (SP={SEG_PARAMS['n_segments']}, "
          f"compactness={SEG_PARAMS['compactness']}, "
          f"S1 weight={S1_FEATURE_WEIGHT})...")
    segments = segment_image(
        blue, green, red, nir, swir1,
        ndvi, ndwi, ndsi, brightness,
        vv_db, vh_db, mean_db,
        SEG_PARAMS,
        s1_weight=S1_FEATURE_WEIGHT,
    )

    # 6. Classify
    print(f"  ↳ Classifying water segments (mode={FUSION_MODE})...")
    water_mask, stats = classify_water_segments(
        segments, ndwi, ndvi, mean_db, SEG_PARAMS, fusion_mode=FUSION_MODE
    )

    # 7. Vectorise 
    print("  ↳ Vectorising water mask...")
    gdf = vectorise_water_mask(water_mask, transform, src_crs, year, stats)

    return gdf


def main():
    conn = connect_openeo()

    all_years_gdfs = []

    for year in tqdm(YEARS, desc="Processing years"):
        try:
            gdf = process_year(conn, year)
            if len(gdf) > 0:
                out_gpkg = os.path.join(OBIA_DIR, f"glacial_lakes_{year}.gpkg")
                gdf.to_file(out_gpkg, driver="GPKG")
                print(f"  ✓ Saved: {out_gpkg}")
                all_years_gdfs.append(gdf)
        except Exception as e:
            print(f"  ✗ Error processing {year}: {e}")
            continue


### Run

In [84]:
main()

Connecting to OpenEO backend: https://openeo.dataspace.copernicus.eu
Authenticated using refresh token.
✓ Connected and authenticated.


Processing years:   0%|          | 0/1 [00:00<?, ?it/s]


  Processing year: 2026  |  Fusion mode: AND
  ↳ [S2] Building OpenEO job for 2026...
  ↳ [S2] Submitting batch job (ID: j-2606231439354843813e383995ee72bb)...
0:00:00 Job 'j-2606231439354843813e383995ee72bb': send 'start'
0:00:02 Job 'j-2606231439354843813e383995ee72bb': queued (progress 0%)
0:00:07 Job 'j-2606231439354843813e383995ee72bb': queued (progress 0%)
0:00:14 Job 'j-2606231439354843813e383995ee72bb': queued (progress 0%)
0:00:22 Job 'j-2606231439354843813e383995ee72bb': queued (progress 0%)
0:00:32 Job 'j-2606231439354843813e383995ee72bb': queued (progress 0%)
0:00:44 Job 'j-2606231439354843813e383995ee72bb': queued (progress 0%)
0:00:59 Job 'j-2606231439354843813e383995ee72bb': running (progress N/A)
0:01:19 Job 'j-2606231439354843813e383995ee72bb': running (progress N/A)
0:01:43 Job 'j-2606231439354843813e383995ee72bb': running (progress N/A)
0:02:13 Job 'j-2606231439354843813e383995ee72bb': running (progress N/A)
0:02:50 Job 'j-2606231439354843813e383995ee72bb': running 

Processing years: 100%|██████████| 1/1 [04:16<00:00, 256.72s/it]
